# Baseline Separation Reduces Shared Context

Notebook 13 showed that differential recovery improves as the noise in two sensors becomes more correlated.

Notebook 17 gives that correlation a physical interpretation:

**As two sensors move farther apart, less environmental noise remains shared.**

This notebook models sensor separation as a baseline distance and asks how common-mode rejection changes when correlation decays with distance.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
FIGURES = ROOT / "figures"
DATA = ROOT / "data"

FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

rng = np.random.default_rng(42)


## Distance-dependent correlation

Let \(d\) be the baseline separation between sensors.

Let \(L\) be a coherence length.

We model the shared-noise correlation as

\[
\rho(d) = e^{-d/L}
\]

Small baselines preserve shared context. Large baselines reduce it.


In [ ]:
baseline = np.linspace(0, 200, 100)

coherence_length = 50

rho = np.exp(-baseline / coherence_length)


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(baseline, rho)

plt.xlabel("Baseline distance")
plt.ylabel("Correlation coefficient (ρ)")

plt.title("Shared context decays with baseline separation")

plt.tight_layout()

plt.savefig(
    FIGURES / "17_correlation_vs_baseline.png",
    dpi=200
)

plt.show()


## Differential recovery versus baseline

Now we reuse the differential sensing model.

As baseline increases, the two sensors share less noise, so differential cancellation becomes less effective.


In [ ]:
n = 1000
t = np.linspace(0, 10, n)

signal = 0.3 * np.sin(2 * np.pi * 0.5 * t)

rmse_baseline = []

for r in rho:

    shared = rng.normal(size=n)

    noise_a = shared

    independent = rng.normal(size=n)

    noise_b = (
        r * shared +
        np.sqrt(1 - r**2) * independent
    )

    local_a = 0.15 * rng.normal(size=n)
    local_b = 0.15 * rng.normal(size=n)

    phi_a = signal / 2 + noise_a + local_a
    phi_b = -signal / 2 + noise_b + local_b

    phi_diff = phi_a - phi_b

    rmse = np.sqrt(
        np.mean((phi_diff - signal) ** 2)
    )

    rmse_baseline.append(rmse)

rmse_baseline = np.array(rmse_baseline)


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(baseline, rmse_baseline)

plt.xlabel("Baseline distance")
plt.ylabel("Differential RMSE")

plt.title("Recovery degrades as baseline reduces shared context")

plt.tight_layout()

plt.savefig(
    FIGURES / "17_baseline_recovery.png",
    dpi=200
)

plt.show()


## Comparing coherence lengths

Different environments can preserve shared structure over different distances.

A longer coherence length means common-mode noise remains correlated over a longer baseline.


In [ ]:
coherence_lengths = [20, 50, 100]

rows = []

plt.figure(figsize=(7, 4))

for L in coherence_lengths:

    rho_L = np.exp(-baseline / L)

    rmse_L = []

    for r in rho_L:

        shared = rng.normal(size=n)

        noise_a = shared

        independent = rng.normal(size=n)

        noise_b = (
            r * shared +
            np.sqrt(1 - r**2) * independent
        )

        local_a = 0.15 * rng.normal(size=n)
        local_b = 0.15 * rng.normal(size=n)

        phi_a = signal / 2 + noise_a + local_a
        phi_b = -signal / 2 + noise_b + local_b

        phi_diff = phi_a - phi_b

        rmse = np.sqrt(
            np.mean((phi_diff - signal) ** 2)
        )

        rmse_L.append(rmse)

    plt.plot(
        baseline,
        rmse_L,
        label=f"L = {L}"
    )

    rows.append({
        "coherence_length": L,
        "rmse_at_baseline_0": rmse_L[0],
        "rmse_at_baseline_100": rmse_L[np.argmin(np.abs(baseline - 100))],
        "rmse_at_baseline_200": rmse_L[-1]
    })

plt.xlabel("Baseline distance")
plt.ylabel("Differential RMSE")

plt.title("Longer coherence length preserves cancellation over distance")

plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "17_coherence_length_comparison.png",
    dpi=200
)

plt.show()


In [ ]:
summary = pd.DataFrame(rows)

summary.to_csv(
    DATA / "17_baseline_separation_summary.csv",
    index=False
)

summary


# Lesson

Baseline length is not just geometry.

It controls which disturbances remain common-mode and which become local.

Differential sensing works best when the measurement design preserves the shared reference while allowing the physical signal to appear in the difference.

Notebook 23 will introduce noise channels and separate common-mode, local, and signal-like disturbances.
